# Session 05 Assignment — Generative Adversarial Networks
### IET CIPHER · NITK · Summer Mentorship Program 2026
---

**Mentor:** Sanskruti &nbsp;|&nbsp; 
> **How to use this notebook**
> - Read each question carefully before writing anything.
> - Fill in code wherever you see `# YOUR CODE HERE`.
> - Fill in written answers wherever you see `YOUR ANSWER HERE`.
> - **Run every cell** before submitting — the notebook must execute top-to-bottom without errors.

---

| | |
|---|---|
| **Name** | *(fill in)* |
| **Roll No.** | *(fill in)* |
| **Date** | *(fill in)* |


---
## Setup — Run This First

Run the cell below before attempting any question. It imports all libraries and sets the random seed.


In [2]:

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from sklearn.cluster import KMeans   # used in Q4

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Device  : {device}")
print(f"   PyTorch : {torch.__version__}")

def show_grid(tensor, nrow=8, title="", figsize=(10, 4)):
    """Display a batch of image tensors as a grid."""
    grid = make_grid(tensor[:nrow*2].view(-1, 1, 28, 28),
                     nrow=nrow, normalize=True, value_range=(-1, 1))
    plt.figure(figsize=figsize)
    plt.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap="gray")
    plt.title(title, fontsize=12, fontweight="bold")
    plt.axis("off")
    plt.tight_layout()
    plt.show()


 Device  : cpu
   PyTorch : 2.12.0+cpu


## Q1 · Architecture Choices &nbsp;&nbsp; 

Answer each part in 2–3 sentences.


### Q2(a) · Tanh vs ReLU/Sigmoid in the Generator &nbsp; 

Why does the Generator use **Tanh** as its final activation instead of ReLU or Sigmoid?


**Your Answer:**
>Tanh normalises the Generator's output to the range [-1, 1], which perfectly matches the scaled range of the input images in the dataset. Furthermore, because Tanh is zero-centered, it provides stronger, more symmetric gradients during backpropagation, which helps the model converge faster and prevents it from getting stuck compared to Sigmoid or ReLU.


### Q2(b) · LeakyReLU in the Discriminator &nbsp;

Why does the Discriminator use **LeakyReLU** instead of standard ReLU?


**Your Answer:**

>LeakyReLU prevents the "dying ReLU" problem by allowing a small, non-zero gradient when the input is negative. This is critical in the Discriminator because it ensures gradients can still flow backwards through the network to the Generator even when the Discriminator's outputs are negative, allowing the Generator to continue learning.

### Q2(c) · DCGAN — Benefits of Convolutions &nbsp; 

DCGAN replaces fully-connected layers with convolutional layers.  
Name **two specific benefits** of this change.


**Your Answer:**

> 1. Spatial Invariance & Feature Extraction: Convolutional layers capture spatial hierarchies and local features (like edges and textures) much better than flattened fully-connected layers, leading to significantly higher-quality images.
> 2. Parameter sharing via convolutional filters drastically reduces the total number of weights in the network, making it faster to train and less prone to overfittin


### Q2(d) · No BatchNorm in Discriminator &nbsp; 

Why is **BatchNorm NOT recommended** inside the Discriminator,  
even though it stabilises the Generator?


**Your Answer:**

> BatchNorm calculates statistics across an entire batch, which creates dependencies between the samples. In the Discriminator, this means it evaluates an image based on the batch's distribution rather than judging the individual image's independent realism, which can introduce batch-level artifacts and destabilize training.

## Q3 · Failure Modes &nbsp;&nbsp; 

For each failure mode: describe **(i) what it looks like in practice**, **(ii) the root cause**, and **(iii) one mitigation strategy**.


### Q3(a) · Mode Collapse &nbsp; 


**Your Answer:**

| | |
|---|---|
| **(i) What it looks like** | The Generator outputs the exact same image (or a very small set of identical images) over and over, regardless of the input noise $z$. |
| **(ii) Root cause** | The Generator discovers a single output that easily fools the Discriminator and stops exploring the rest of the data distribution, optimizing only for that specific point. |
| **(iii) Mitigation** | Use Minibatch Discrimination, Unrolled GANs, or switch to a Wasserstein GAN (WGAN) loss to encourage diversity and penalize lack of variance. |


### Q3(b) · Vanishing Gradients in the Generator &nbsp; 


**Your Answer:**

| | |
|---|---|
| **(i) What it looks like** | The Generator stops improving, its loss plateaus, and it produces pure noise while the Discriminator's loss drops to zero. |
| **(ii) Root cause** | The Discriminator becomes too "perfect" too quickly, easily distinguishing real from fake. It outputs probabilities near 0 or 1, meaning the gradients passed back to the Generator approach zero. |
| **(iii) Mitigation** | Use the non-saturating heuristic loss for the Generator (maximizing $\log(D(G(z)))$) or use Wasserstein GAN (WGAN) which provides meaningful linear gradients even when the critic is optimal. |

---


## Q4 · Build  Vanilla GAN &nbsp;&nbsp; 
You will implement a GAN from scratch and train it on MNIST.  
Follow the exact architecture below — **do not change layer sizes or activations**.

| Component | Architecture |
|-----------|-------------|
| **Generator input** | Noise $z \sim \mathcal{N}(0,1)$, `dim = 100` |
| **Generator hidden** | `Linear(100→256)` + `BatchNorm1d` + `LeakyReLU(0.2)` → `Linear(256→512)` + `BatchNorm1d` + `LeakyReLU(0.2)` → `Linear(512→1024)` + `BatchNorm1d` + `LeakyReLU(0.2)` |
| **Generator output** | `Linear(1024→784)` + `Tanh` |
| **Discriminator input** | Flat image, `dim = 784` |
| **Discriminator hidden** | `Linear(784→512)` + `LeakyReLU(0.2)` + `Dropout(0.3)` → `Linear(512→256)` + `LeakyReLU(0.2)` + `Dropout(0.3)` |
| **Discriminator output** | `Linear(256→1)` + `Sigmoid` |
| **Optimiser (both)** | `Adam`, `lr=0.0002`, `β₁=0.5` |
| **Label smoothing** | Real labels = `0.9`, Fake labels = `0.0` |
| **Batch size** | `128` |
| **Epochs** | `50` |

> 


### Step 1 · Load MNIST &nbsp; *(provided — just run)*


In [3]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
BATCH_SIZE = 128
IMAGE_SIZE = 784     # 28 × 28
LATENT_DIM = 100
LR         = 0.0002
EPOCHS     = 50

# ── Dataset ───────────────────────────────────────────────────────────────────
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])   # scale to [-1, 1]
])

train_dataset = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)
dataloader    = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Train: {len(train_dataset):,} images | Test: {len(test_dataset):,} images")
print(f"Batches per epoch: {len(dataloader)}")


100.0%
100.0%
100.0%
100.0%

Train: 60,000 images | Test: 10,000 images
Batches per epoch: 469


### Step 2 · Implement the Generator &nbsp; `[5 marks]`

Complete the `Generator` class. Use the architecture from the table above.


In [4]:
class Generator(nn.Module):
    def __init__(self, latent_dim, image_size):
        super().__init__()

        # ── TODO: Build self.model as an nn.Sequential ────────────────────────
        # Layer order:
        #   block 1: Linear(latent_dim → 256) + BatchNorm1d(256) + LeakyReLU(0.2)
        #   block 2: Linear(256 → 512)        + BatchNorm1d(512) + LeakyReLU(0.2)
        #   block 3: Linear(512 → 1024)       + BatchNorm1d(1024)+ LeakyReLU(0.2)
        #   output : Linear(1024 → image_size) + Tanh

        self.model = nn.Sequential(
            nn.Linear(latent_dim, 256),
            nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2),
            
            nn.Linear(256, 512),
            nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),
            
            nn.Linear(512, 1024),
            nn.BatchNorm1d(1024),
            nn.LeakyReLU(0.2),
            
            nn.Linear(1024, image_size),
            nn.Tanh()
        )
        

    def forward(self, z):
        return self.model(z)


G = Generator(LATENT_DIM, IMAGE_SIZE).to(device)
z_test = torch.randn(4, LATENT_DIM).to(device)
out    = G(z_test)
assert out.shape == (4, IMAGE_SIZE), f"Expected (4, 784), got {out.shape}"
print("Generator output shape:", out.shape)
print("   Output range:", round(out.min().item(), 3), "→", round(out.max().item(), 3), "(should be ≈ [-1, 1])")
print(G)


Generator output shape: torch.Size([4, 784])
   Output range: -0.902 → 0.91 (should be ≈ [-1, 1])
Generator(
  (model): Sequential(
    (0): Linear(in_features=100, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): LeakyReLU(negative_slope=0.2)
    (3): Linear(in_features=256, out_features=512, bias=True)
    (4): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): LeakyReLU(negative_slope=0.2)
    (6): Linear(in_features=512, out_features=1024, bias=True)
    (7): BatchNorm1d(1024, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (8): LeakyReLU(negative_slope=0.2)
    (9): Linear(in_features=1024, out_features=784, bias=True)
    (10): Tanh()
  )
)


### Step 3 · Implement the Discriminator &nbsp; `[5 marks]`

Complete the `Discriminator` class. Use the architecture from the table above.


In [5]:
class Discriminator(nn.Module):
    def __init__(self, image_size):
        super().__init__()

        # ── TODO: Build self.model as an nn.Sequential ────────────────────────
        # Layer order:
        #   block 1: Linear(image_size → 512) + LeakyReLU(0.2) + Dropout(0.3)
        #   block 2: Linear(512 → 256)        + LeakyReLU(0.2) + Dropout(0.3)
        #   output : Linear(256 → 1)          + Sigmoid

        self.model = nn.Sequential(
            nn.Linear(image_size, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            
            nn.Linear(256, 1),
            nn.Sigmoid()
        
        )

    def forward(self, x):
        return self.model(x)


D = Discriminator(IMAGE_SIZE).to(device)
x_test = torch.randn(4, IMAGE_SIZE).to(device)
pred   = D(x_test)
assert pred.shape == (4, 1), f"Expected (4, 1), got {pred.shape}"
print("Discriminator output shape:", pred.shape)
print("   Output range:", round(pred.min().item(), 3), "→", round(pred.max().item(), 3), "(should be in [0, 1])")
print(D)


Discriminator output shape: torch.Size([4, 1])
   Output range: 0.499 → 0.567 (should be in [0, 1])
Discriminator(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): LeakyReLU(negative_slope=0.2)
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=512, out_features=256, bias=True)
    (4): LeakyReLU(negative_slope=0.2)
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=256, out_features=1, bias=True)
    (7): Sigmoid()
  )
)


### Step 4 · Define Loss & Optimisers &nbsp; `[2 marks]`

Complete the cell below:
- Use `nn.BCELoss()` as the loss function.
- Create an Adam optimiser for G and a separate Adam optimiser for D.
- Use `lr=0.0002` and `betas=(0.5, 0.999)`.


In [7]:
criterion = nn.BCELoss()   # YOUR CODE HERE — replace None with nn.BCELoss()

optimizer_G = optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))
optimizer_D = optim.Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))
def real_labels(n): return torch.full((n, 1), 0.9).to(device)
def fake_labels(n): return torch.zeros(n, 1).to(device)

fixed_noise = torch.randn(64, LATENT_DIM).to(device)

assert criterion is not None,   "criterion is not defined!"
assert optimizer_G is not None, "optimizer_G is not defined!"
assert optimizer_D is not None, "optimizer_D is not defined!"
print(" Loss and optimisers ready")


 Loss and optimisers ready
